In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PASTA_RAIZ = Path.cwd().resolve().parent
PASTA_ANALYTICS = PASTA_RAIZ / "data" / "analytics"

caminho_integrado = PASTA_ANALYTICS / "enem_2022_ibge_integrado.csv"

df_enem_ibge_2022 = pd.read_csv(
    caminho_integrado,
    sep=";",
    encoding="utf-8"
)

print("Base integrada carregada:")
print(df_enem_ibge_2022.shape)

display(df_enem_ibge_2022.head())

Base integrada carregada:
(2504014, 27)


,NU_ANO,SG_UF_PROVA,TP_SEXO,TP_FAIXA_ETARIA,TP_ESCOLA,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,CO_MUNICIPIO_PROVA,...,NU_NOTA_REDACAO,MEDIA_GERAL,RENDA_FAMILIAR,ACESSO_INTERNET,TIPO_ESCOLA,DEPENDENCIA_ESCOLA,LOCALIZACAO_ESCOLA,NO_MUNICIPIO_IBGE,POPULACAO_MUNICIPIO,PORTE_MUNICIPIO
0,2022,BA,F,5,1,Não informado,Não informado,Não informado,Não informado,2925758,...,760.0,558.24,Até 1 salário mínimo,Sim,Não respondeu,Não informado,Não informado,Presidente Tancredo Neves - Ba,27734,20 mil a 100 mil
1,2022,ES,M,6,1,Não informado,Não informado,Não informado,Não informado,3201308,...,320.0,394.62,Nenhuma renda,Sim,Não respondeu,Não informado,Não informado,Cariacica - Es,353491,100 mil a 500 mil
2,2022,RJ,F,6,1,Não informado,Não informado,Não informado,Não informado,3304904,...,440.0,414.10,Até 1 salário mínimo,Sim,Não respondeu,Não informado,Não informado,São Gonçalo - Rj,896744,500 mil a 1 milhão
3,2022,PE,F,4,1,Não informado,Não informado,Não informado,Não informado,2601201,...,360.0,438.10,Até 1 salário mínimo,Sim,Não respondeu,Não informado,Não informado,Arcoverde - Pe,77742,20 mil a 100 mil
4,2022,SE,F,2,3,Não informado,Não informado,Não informado,Não informado,2804508,...,940.0,576.70,Até 1 salário mínimo,Sim,Privada,Não informado,Não informado,Nossa Senhora Da Glória - Se,41212,20 mil a 100 mil


### Criando novas variáveis

In [3]:
df_transformado = df_enem_ibge_2022.copy()

colunas_notas = [
    "NU_NOTA_CN",
    "NU_NOTA_CH",
    "NU_NOTA_LC",
    "NU_NOTA_MT",
    "NU_NOTA_REDACAO"
]

# Garante que as notas estejam como número
for coluna in colunas_notas:
    df_transformado[coluna] = pd.to_numeric(
        df_transformado[coluna],
        errors="coerce"
    )

# Média geral
df_transformado["MEDIA_GERAL"] = df_transformado[colunas_notas].mean(axis=1)

# Média apenas das provas objetivas
df_transformado["MEDIA_OBJETIVAS"] = df_transformado[
    [
        "NU_NOTA_CN",
        "NU_NOTA_CH",
        "NU_NOTA_LC",
        "NU_NOTA_MT"
    ]
].mean(axis=1)

# Diferença entre redação e média das objetivas
df_transformado["DIF_REDACAO_OBJETIVAS"] = (
    df_transformado["NU_NOTA_REDACAO"] - df_transformado["MEDIA_OBJETIVAS"]
)

# Classificação de desempenho
df_transformado["NIVEL_DESEMPENHO"] = pd.cut(
    df_transformado["MEDIA_GERAL"],
    bins=[0, 400, 500, 600, 700, 1000],
    labels=[
        "Muito baixo",
        "Baixo",
        "Médio",
        "Alto",
        "Muito alto"
    ]
)

display(df_transformado[
    [
        "MEDIA_GERAL",
        "MEDIA_OBJETIVAS",
        "NU_NOTA_REDACAO",
        "DIF_REDACAO_OBJETIVAS",
        "NIVEL_DESEMPENHO"
    ]
].head())

,MEDIA_GERAL,MEDIA_OBJETIVAS,NU_NOTA_REDACAO,DIF_REDACAO_OBJETIVAS,NIVEL_DESEMPENHO
0,558.24,507.800,760.0,252.200,Médio
1,394.62,413.275,320.0,-93.275,Muito baixo
2,414.10,407.625,440.0,32.375,Baixo
3,438.10,457.625,360.0,-97.625,Baixo
4,576.70,485.875,940.0,454.125,Médio


Agregado por renda Familia

In [4]:
ordem_renda = [
    "Nenhuma renda",
    "Até 1 salário mínimo",
    "1 a 1,5 salários",
    "1,5 a 2 salários",
    "2 a 2,5 salários",
    "2,5 a 3 salários",
    "3 a 4 salários",
    "4 a 5 salários",
    "5 a 6 salários",
    "6 a 7 salários",
    "7 a 8 salários",
    "8 a 9 salários",
    "9 a 10 salários",
    "10 a 12 salários",
    "12 a 15 salários",
    "15 a 20 salários",
    "Acima de 20 salários",
    "Não informado"
]

agregado_renda = (
    df_transformado
    .groupby("RENDA_FAMILIAR", as_index=False)
    .agg(
        TOTAL_PARTICIPANTES=("MEDIA_GERAL", "count"),
        MEDIA_GERAL=("MEDIA_GERAL", "mean"),
        MEDIA_REDACAO=("NU_NOTA_REDACAO", "mean"),
        MEDIA_OBJETIVAS=("MEDIA_OBJETIVAS", "mean"),
        DIF_MEDIA_REDACAO_OBJETIVAS=("DIF_REDACAO_OBJETIVAS", "mean")
    )
)

agregado_renda["RENDA_FAMILIAR"] = pd.Categorical(
    agregado_renda["RENDA_FAMILIAR"],
    categories=ordem_renda,
    ordered=True
)

agregado_renda = agregado_renda.sort_values("RENDA_FAMILIAR")

display(agregado_renda)

,RENDA_FAMILIAR,TOTAL_PARTICIPANTES,MEDIA_GERAL,MEDIA_REDACAO,MEDIA_OBJETIVAS,DIF_MEDIA_REDACAO_OBJETIVAS
16,Nenhuma renda,129574,477.785239,515.445208,469.341799,46.002478
15,Até 1 salário mínimo,680600,497.704595,553.539498,484.502276,68.933495
0,"1 a 1,5 salários",397091,522.659347,590.949820,506.131540,84.721717
1,"1,5 a 2 salários",295407,536.182449,612.549495,517.513634,94.945095
5,"2 a 2,5 salários",206495,549.450145,635.185716,528.286833,106.814542
6,"2,5 a 3 salários",121832,561.200896,653.501520,538.367101,115.039419
7,3 a 4 salários,153631,570.366334,669.753251,545.629080,124.019328
8,4 a 5 salários,153525,582.491439,690.806829,555.416462,135.267203
9,5 a 6 salários,65447,593.753484,708.767691,564.863240,143.755109
10,6 a 7 salários,46294,600.079735,719.386962,570.183744,149.038246


Agregado por acesso a internet

In [5]:
agregado_internet = (
    df_transformado
    .groupby("ACESSO_INTERNET", as_index=False)
    .agg(
        TOTAL_PARTICIPANTES=("MEDIA_GERAL", "count"),
        MEDIA_GERAL=("MEDIA_GERAL", "mean"),
        MEDIA_REDACAO=("NU_NOTA_REDACAO", "mean"),
        MEDIA_OBJETIVAS=("MEDIA_OBJETIVAS", "mean")
    )
    .sort_values("MEDIA_GERAL", ascending=False)
)

display(agregado_internet)

,ACESSO_INTERNET,TOTAL_PARTICIPANTES,MEDIA_GERAL,MEDIA_REDACAO,MEDIA_OBJETIVAS
1,Sim,2298919,544.029594,626.974636,523.626965
0,Não,205095,482.161222,523.160928,472.830662


Agregado por tipo de escola

In [6]:
agregado_tipo_escola = (
    df_transformado
    .groupby("TIPO_ESCOLA", as_index=False)
    .agg(
        TOTAL_PARTICIPANTES=("MEDIA_GERAL", "count"),
        MEDIA_GERAL=("MEDIA_GERAL", "mean"),
        MEDIA_REDACAO=("NU_NOTA_REDACAO", "mean"),
        MEDIA_OBJETIVAS=("MEDIA_OBJETIVAS", "mean")
    )
    .sort_values("MEDIA_GERAL", ascending=False)
)

display(agregado_tipo_escola)

,TIPO_ESCOLA,TOTAL_PARTICIPANTES,MEDIA_GERAL,MEDIA_REDACAO,MEDIA_OBJETIVAS
1,Privada,201037,606.711452,751.272068,570.428984
0,Não respondeu,1490185,543.701272,623.365812,523.955066
2,Pública,812792,513.516246,576.635138,498.631718


Agregado por porte de municipio

In [7]:
agregado_porte_municipio = (
    df_transformado
    .groupby("PORTE_MUNICIPIO", as_index=False)
    .agg(
        TOTAL_PARTICIPANTES=("MEDIA_GERAL", "count"),
        MEDIA_GERAL=("MEDIA_GERAL", "mean"),
        MEDIA_REDACAO=("NU_NOTA_REDACAO", "mean"),
        MEDIA_OBJETIVAS=("MEDIA_OBJETIVAS", "mean"),
        POPULACAO_MEDIA=("POPULACAO_MUNICIPIO", "mean")
    )
    .sort_values("POPULACAO_MEDIA")
)

display(agregado_porte_municipio)

,PORTE_MUNICIPIO,TOTAL_PARTICIPANTES,MEDIA_GERAL,MEDIA_REDACAO,MEDIA_OBJETIVAS,POPULACAO_MEDIA
4,Até 20 mil,99925,504.720749,573.022503,488.296546,1.563586e+04
1,20 mil a 100 mil,776563,520.640812,595.321027,502.471548,5.106819e+04
0,100 mil a 500 mil,764586,543.084482,624.753550,523.010766,2.555288e+05
2,500 mil a 1 milhão,274592,555.539581,643.127615,533.912829,7.369969e+05
3,Acima de 1 milhão,588348,555.866083,637.099302,535.843474,4.232387e+06


Agregado por município da prova

In [8]:
agregado_municipio = (
    df_transformado
    .groupby(
        [
            "CO_MUNICIPIO_PROVA",
            "NO_MUNICIPIO_PROVA",
            "SG_UF_PROVA",
            "PORTE_MUNICIPIO"
        ],
        as_index=False
    )
    .agg(
        TOTAL_PARTICIPANTES=("MEDIA_GERAL", "count"),
        MEDIA_GERAL=("MEDIA_GERAL", "mean"),
        MEDIA_REDACAO=("NU_NOTA_REDACAO", "mean"),
        MEDIA_OBJETIVAS=("MEDIA_OBJETIVAS", "mean"),
        POPULACAO_MUNICIPIO=("POPULACAO_MUNICIPIO", "first")
    )
    .sort_values("TOTAL_PARTICIPANTES", ascending=False)
)

display(agregado_municipio.head(20))

,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,SG_UF_PROVA,PORTE_MUNICIPIO,TOTAL_PARTICIPANTES,MEDIA_GERAL,MEDIA_REDACAO,MEDIA_OBJETIVAS,POPULACAO_MUNICIPIO
1314,3550308,São Paulo,SP,Acima de 1 milhão,108054,565.093302,640.287616,546.387563,11451999
1116,3304557,Rio de Janeiro,RJ,Acima de 1 milhão,77449,562.714163,644.793363,542.623428,6211223
371,2304400,Fortaleza,CE,Acima de 1 milhão,51449,551.624974,638.643922,530.131623,2428708
1746,5300108,Brasília,DF,Acima de 1 milhão,47566,555.713560,632.347727,536.946782,2817381
115,1501402,Belém,PA,Acima de 1 milhão,44088,528.001129,612.707883,507.272130,1303403
815,2927408,Salvador,BA,Acima de 1 milhão,38943,547.354898,630.144898,526.850467,2417678
872,3106200,Belo Horizonte,MG,Acima de 1 milhão,36847,588.716133,684.416880,564.994752,2315560
611,2611606,Recife,PE,Acima de 1 milhão,30962,555.671600,638.759926,535.184976,1488920
73,1302603,Manaus,AM,Acima de 1 milhão,30947,511.121344,558.644024,500.206032,2063689
285,2111300,São Luís,MA,Acima de 1 milhão,28950,530.583814,617.941523,508.778330,1037775


Resumo geral para cards do Dash

In [9]:
resumo_geral = pd.DataFrame({
    "TOTAL_PARTICIPANTES": [len(df_transformado)],
    "MEDIA_GERAL": [df_transformado["MEDIA_GERAL"].mean()],
    "MEDIA_REDACAO": [df_transformado["NU_NOTA_REDACAO"].mean()],
    "MEDIA_OBJETIVAS": [df_transformado["MEDIA_OBJETIVAS"].mean()],
    "PERCENTUAL_COM_INTERNET": [
        df_transformado["ACESSO_INTERNET"].eq("Sim").mean() * 100
    ],
    "RENDA_MAIS_COMUM": [
        df_transformado["RENDA_FAMILIAR"].mode().iloc[0]
    ],
    "TOTAL_MUNICIPIOS_PROVA": [
        df_transformado["CO_MUNICIPIO_PROVA"].nunique()
    ]
})

display(resumo_geral)

,TOTAL_PARTICIPANTES,MEDIA_GERAL,MEDIA_REDACAO,MEDIA_OBJETIVAS,PERCENTUAL_COM_INTERNET,RENDA_MAIS_COMUM,TOTAL_MUNICIPIOS_PROVA
0,2504014,538.962173,618.4797,519.466418,91.809351,Até 1 salário mínimo,1747


Salvando bases transformadas e agregadas para uso no DASH

In [10]:
df_transformado.to_csv(
    PASTA_ANALYTICS / "enem_2022_ibge_transformado.csv",
    sep=";",
    encoding="utf-8",
    index=False
)

resumo_geral.to_csv(
    PASTA_ANALYTICS / "resumo_geral_2022.csv",
    sep=";",
    encoding="utf-8",
    index=False
)

agregado_renda.to_csv(
    PASTA_ANALYTICS / "agregado_renda_2022.csv",
    sep=";",
    encoding="utf-8",
    index=False
)

agregado_internet.to_csv(
    PASTA_ANALYTICS / "agregado_internet_2022.csv",
    sep=";",
    encoding="utf-8",
    index=False
)

agregado_tipo_escola.to_csv(
    PASTA_ANALYTICS / "agregado_tipo_escola_2022.csv",
    sep=";",
    encoding="utf-8",
    index=False
)

agregado_porte_municipio.to_csv(
    PASTA_ANALYTICS / "agregado_porte_municipio_2022.csv",
    sep=";",
    encoding="utf-8",
    index=False
)

agregado_municipio.to_csv(
    PASTA_ANALYTICS / "agregado_municipio_2022.csv",
    sep=";",
    encoding="utf-8",
    index=False
)

print("Arquivos transformados salvos em data/analytics.")

Arquivos transformados salvos em data/analytics.
